In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from datetime import date, datetime, timedelta
from scipy.stats import ks_2samp, mannwhitneyu#, anderson_ksamp
sns.set_theme(style="ticks")
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Arial'
plt.rcParams['mathtext.it'] = 'Arial:italic'

# parameters

In [ ]:
yeari, yearf = '2024', '2024'
weeki, weekf = '18', '31'

In [ ]:
di = datetime.strptime(f'{yeari}-{weeki}-1', "%Y-%W-%w").date()
df = datetime.strptime(f'{yearf}-{weekf}-1', "%Y-%W-%w").date() + timedelta(6)
ds = [di+timedelta(dt) for dt in range((df-di).days+1)]
daylist = ds
print(di, 'until', df)

In [ ]:
cdef = 'tl7_10m'# 'tl5_10m' 'tl6_10m' 'tl7_10m' 'tl8_10m' 'tl8_60m'
cdef_alt = '16m_10min'# tl5: 62 ... tl7: 16   tl8: 8

# load data

In [ ]:
# load mass event data
match_data = pd.read_csv('data/metadata/event_data.csv')
match_data['day'] = [d.date() for d in pd.to_datetime(match_data.day)]

In [ ]:
data = pd.read_csv('data/fig5/follow_didpairs.csv')
data['day'] = [d.date() for d in pd.to_datetime(data.day)]
data['stime'] = pd.to_datetime(data.stime)
dmin = data.day.min()
data['tt2'] = data.day.apply(lambda d: (d-dmin).days)*24*6 + data.stime.dt.hour*6 + (data.stime.dt.minute//10)
print(data.tt2.min(), data.tt2.max())

In [ ]:
(data.day.max()-dmin).days*24*6 + (24-1)*6 + (6-1)

# analyses

In [ ]:
# stability: 0 = random pair, 1 = stable pair
randstab = pd.DataFrame(data.groupby('pair').day.apply(lambda x: 0 if len(set(x))==1 else 1)).reset_index()
randstab = randstab.rename(columns={'tt':'stability','day':'stability','tt2':'stability'})
randstab

In [ ]:
data = pd.read_csv('data/fig5/follow_didpairs_cities.csv')
data['day'] = [d.date() for d in pd.to_datetime(data.day)]
data['stime'] = pd.to_datetime(data.stime)
dmin = data.day.min()
data['tt2'] = data.day.apply(lambda d: (d-dmin).days)*24*6 + data.stime.dt.hour*6 + (data.stime.dt.minute//10)
print(data.tt2.min(), data.tt2.max())

In [ ]:
match_base = match_data[['day','city','area_id']].copy(deep=True)
match_base['day'] = [d-timedelta(7) for d in match_base.day]
match_base['area_id'] = np.nan
len(match_base)

In [ ]:
dist_cmp_pos = data.merge(match_data[match_data.day>=di+timedelta(7)][['day','city','area_id']], on=['day','city'])
dist_cmp_pos['event'] = True
dist_cmp_neg = data.merge(match_base[['day','city','area_id']], on=['day','city'])
dist_cmp_neg['event'] = False
dist_cmp = pd.concat([dist_cmp_pos, dist_cmp_neg]).merge(randstab, on='pair')
daily_dur = dist_cmp.groupby(['day','pair']).tt2.apply(lambda x: len(set(x))).reset_index().rename(columns={'tt2':'dur'})
dist_cmp = dist_cmp.merge(daily_dur, on=['day','pair'])
dist_cmp

In [ ]:
dist_cmp['dhome1'], dist_cmp['dhome2'] = np.where(dist_cmp['dhome1'] > dist_cmp['dhome2'],\
                                                  (dist_cmp['dhome2'], dist_cmp['dhome1']),\
                                                  (dist_cmp['dhome1'], dist_cmp['dhome2']))
dist_cmp['dhome1'] /= 1e3
dist_cmp['dhome2'] /= 1e3
dist_cmp['dhometohome'] /= 1e3
dist_cmp['dur'] *= 1e1
dist_cmp

In [ ]:
dist_cmp[dist_cmp.city=='Gelsenkirchen'].dhome2.max()

In [ ]:
dist_cmp.dhome2.max()

In [ ]:
dist_cmp[dist_cmp.city=='Gelsenkirchen'].dhometohome.max()

In [ ]:
dist_cmp.dhometohome.max()

In [ ]:
len(set(dist_cmp[(dist_cmp.city=='Gelsenkirchen') & (dist_cmp.event==True)].day))

In [ ]:
len(set(dist_cmp[(dist_cmp.city=='Gelsenkirchen') & (dist_cmp.event==False)].day))

In [ ]:
len(set(dist_cmp[dist_cmp.event==True].day))

In [ ]:
len(set(dist_cmp[dist_cmp.event==False].day))

In [ ]:
days_here = set(dist_cmp[dist_cmp.event==False].day)
len(days_here)

In [ ]:
len(set([(city, day) for city, day, event in zip(dist_cmp.city, dist_cmp.day, dist_cmp.event) if event==True]))

In [ ]:
len(set([(city, day) for city, day, event in zip(dist_cmp.city, dist_cmp.day, dist_cmp.event) if event==False]))

## 10 cities

In [ ]:
titles = ['non-event', 'event']
g = sns.relplot(
    data=dist_cmp, x="dhome1", y="dhome2",#[dist_cmp.day.isin(days_here)]
    col="event", hue="stability", size="dur", col_order=[False,True],# style="day",
    kind="scatter", alpha=1.,
    palette=sns.husl_palette(2),#"ch:r=-.2,d=.3_r",
    #hue_order=clarity_ranking,
    sizes=(.1, 25), linewidth=0,# (.1, 25)
)
for ax, tit in zip(g.axes.flat, titles):
    ax.set_xscale('symlog', linthresh=1e-1)
    ax.set_yscale('symlog', linthresh=1e-1)
    ax.set_xlim([-1e-2, 1.2e3])
    ax.set_ylim([-1e-2, 1.2e3])
    ax.set_xlabel(r'contact-to-home distance device 1 $d_1$ [km]')
    ax.set_ylabel(r'contact-to-home distance device 2 $d_2$ [km]')
    ax.set_aspect('equal')
    ax.set_title(tit)
    ax.grid(False)
#g.add_legend(title='category of contact')#, bbox_to_anchor=(1, 1), loc="center left")
lg = g._legend
for tx in lg.texts:
    if tx.get_text() == 'stability':
        tx.set_text('category')
    elif tx.get_text() == '1':
        tx.set_text('recurrent')
    elif tx.get_text() == '0':
        tx.set_text('random')
    elif tx.get_text() == 'dur':
        tx.set_text('duration [min]')
g.tight_layout()

plt.savefig(f'plots/figs10/stable_random_homedist_germany.jpg', bbox_inches='tight', dpi=300)
##plt.savefig(f'plots/figs10/stable_random_homedist_germany.pdf', bbox_inches='tight')
plt.show()

In [ ]:
dhhstat = dist_cmp[['event','stability','dhometohome']].copy(deep=True)
dhhstat['stability'] = ['recurrent' if sta==1 else 'random' for sta in dhhstat.stability]
dhhstat['dhometohome_log'] = dhhstat.dhometohome.apply(lambda x: np.log10(1e-3+x))
print(dhhstat.dhometohome.min(), dhhstat.dhometohome.max())
dhhstat

In [ ]:
# Initialize a grid of plots with an Axes for each walk
grid = sns.FacetGrid(dhhstat, col="event", hue="stability", hue_order=['random','recurrent'], palette=sns.husl_palette(2),#sns.color_palette(),#"tab20c",
                     col_wrap=2, height=2.5, aspect=1, legend_out=True)

# Draw a line plot to show the trajectory of each random walk
grid.map(sns.histplot, "dhometohome", binwidth=.25, binrange=(-4.,3.), multiple="stack",edgecolor=".3", linewidth=.5, log_scale=True,)
grid.set(xlim=[1e-4,1e3], xlabel="home-to-home dist [km]", ylabel="observed contacts")
grid.set(xticks=[1e-3,1e-2,1e-1,1e0,1e1,1e2])

# legend
grid.add_legend(title='category')

# Custom formatter: show decimals only when needed
def smart_format(x, _):
    return f"{x:.2f}".rstrip('0').rstrip('.') if x < 10 else f"{int(x)}"

for ax, tit in zip(grid.axes.flat, ['non-event', 'event']):
    ax.set_title(tit)
    #ax.xaxis.set_major_formatter(formatter)
    #ax.xaxis.set_major_formatter(mpl.ticker.FormatStrFormatter('%.2f'))
    #ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(smart_format))

plt.savefig(f'plots/fig5/stable_random_homehomedist_germany_linear.jpg', bbox_inches='tight', dpi=300)
plt.savefig(f'plots/fig5/stable_random_homehomedist_germany_linear.pdf', bbox_inches='tight')
plt.show()

In [ ]:
stab_here = 'recurrent'
#stab_here = 'random'
data1 = dhhstat[(~dhhstat.dhometohome.isna()) & (dhhstat.stability==stab_here) & (dhhstat.event==False)].dhometohome_log.tolist()
data2 = dhhstat[(~dhhstat.dhometohome.isna()) & (dhhstat.stability==stab_here) & (dhhstat.event==True)].dhometohome_log.tolist()
print(len(data1), len(data2))

print()
 
statistic, p_value = ks_2samp(data1, data2)
print(f"KS statistic: {statistic:.3f}, p-value: {p_value}")# {p_value:.4f}")

print()

statistic, p_value = mannwhitneyu(data1, data2, alternative='two-sided')
print(f"Mann–Whitney U statistic: {statistic}, p-value: {p_value}")# {p_value:.4f}")

#print()
#
#result = anderson_ksamp([data1, data2])
#print(f"AD statistic: {result.statistic}, p-value: {result.significance_level:.4f}")

In [ ]:
catstat = dist_cmp.groupby(['day','event','stability']).pair.apply(lambda x: len(x)).reset_index().drop(columns=['day'])# len(set(x))
catstat['stability'] = ['recurrent' if sta==1 else 'random' for sta in catstat.stability]
catstat['pair'] = catstat.pair.astype(float)
catstat['x'] = catstat.stability.map({'random':1,'recurrent':0})
catstat['y'] = catstat.event.map({True:1,False:0})
print(catstat.pair.sum())
catstat

In [ ]:
# Initialize a grid of plots with an Axes for each walk
grid = sns.FacetGrid(catstat, col="event", hue="stability", hue_order=['random','recurrent'], palette=sns.husl_palette(2),#sns.color_palette(),#"tab20c",
                     col_wrap=2, height=2.5, aspect=1, legend_out=True)

# Draw a line plot to show the trajectory of each random walk
grid.map(sns.violinplot, "x", "pair", orient="v",
               split=False, bw_adjust=1, cut=.1, linewidth=1, order=[0,1],
               fill=False, inner='box', gap=.25, density_norm='width',
               legend=False,
               inner_kws=dict(box_width=7.5, whis_width=0))
grid.set(xlabel="", ylabel="observed contacts")
grid.set(xticks=[0,1], xticklabels=['recurrent', 'random'])

# Custom formatter: show decimals only when needed
def smart_format(x, _):
    return f"{x:.2f}".rstrip('0').rstrip('.') if x < 10 else f"{int(x)}"

for ax, tit in zip(grid.axes.flat, ['non-event', 'event']):
    ax.set_title(tit)
    #ax.xaxis.set_major_formatter(formatter)
    #ax.xaxis.set_major_formatter(mpl.ticker.FormatStrFormatter('%.2f'))
    #ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(smart_format))

plt.show()

In [ ]:
# Initialize a grid of plots with an Axes for each walk
grid = sns.FacetGrid(catstat, col="stability", hue="stability", hue_order=['random','recurrent'], palette=sns.husl_palette(2),#sns.color_palette(),#"tab20c",
                     col_order=['recurrent','random'], col_wrap=2, height=2.5, aspect=1, legend_out=True)

# Draw a line plot to show the trajectory of each random walk
grid.map(sns.violinplot, "y", "pair", orient="v",
               split=False, bw_adjust=1, cut=.1, linewidth=1, order=[0,1],
               fill=False, inner='box', gap=.25, density_norm='width',
               legend=False,
               inner_kws=dict(box_width=7.5, whis_width=0))
grid.set(xlabel="", ylabel="observed contacts")
grid.set(xticks=[0,1], xticklabels=['non-event','event'])

# Custom formatter: show decimals only when needed
def smart_format(x, _):
    return f"{x:.2f}".rstrip('0').rstrip('.') if x < 10 else f"{int(x)}"

for ax, tit in zip(grid.axes.flat, ['stable', 'random']):
    ax.set_title(tit)
    #ax.xaxis.set_major_formatter(formatter)
    #ax.xaxis.set_major_formatter(mpl.ticker.FormatStrFormatter('%.2f'))
    #ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(smart_format))

plt.savefig(f'plots/fig5/stable_random_homedist_germany_histo.jpg', bbox_inches='tight', dpi=300)
plt.savefig(f'plots/fig5/stable_random_homedist_germany_histo.pdf', bbox_inches='tight')
plt.show()

In [ ]:
stab_here = 'recurrent'
#stab_here = 'random'
data1 = catstat[(catstat.stability==stab_here) & (catstat.event==False)].pair.tolist()
data2 = catstat[(catstat.stability==stab_here) & (catstat.event==True)].pair.tolist()
print(len(data1), len(data2))

print()
 
statistic, p_value = ks_2samp(data1, data2)
print(f"KS statistic: {statistic:.3f}, p-value: {p_value:.4f}")

print()

statistic, p_value = mannwhitneyu(data1, data2, alternative='two-sided')
print(f"Mann–Whitney U statistic: {statistic}, p-value: {p_value:.4f}")

#print()
#
#result = anderson_ksamp([data1, data2])
#print(f"AD statistic: {result.statistic}, p-value: {result.significance_level:.4f}")

## 1 city

In [ ]:
titles = ['non-event', 'event']
g = sns.relplot(
    data=dist_cmp[dist_cmp.city=='Gelsenkirchen'], x="dhome1", y="dhome2",
    col="event", hue="stability", size="dur", col_order=[False,True],# style="day",
    kind="scatter", alpha=1.,
    palette=sns.husl_palette(2),#"ch:r=-.2,d=.3_r",
    #hue_order=clarity_ranking,
    sizes=(1, 250), linewidth=0,# (.1,25)
)
for ax, tit in zip(g.axes.flat, titles):
    ax.set_xscale('symlog', linthresh=1e-1)
    ax.set_yscale('symlog', linthresh=1e-1)
    ax.set_xlim([-1e-2, 2e3])
    ax.set_ylim([-1e-2, 2e3])
    ax.set_xlabel(r'contact-to-home distance device 1 $d_1$ [km]')
    ax.set_ylabel(r'contact-to-home distance device 2 $d_2$ [km]')
    ax.set_aspect('equal')
    ax.set_title(tit)
    ax.grid(False)
#g.add_legend(title='category of contact')#, bbox_to_anchor=(1, 1), loc="center left")
lg = g._legend
for tx in lg.texts:
    if tx.get_text() == 'stability':
        tx.set_text('category')
    elif tx.get_text() == '1':
        tx.set_text('recurrent')
    elif tx.get_text() == '0':
        tx.set_text('random')
    elif tx.get_text() == 'dur':
        tx.set_text('duration [min]')
g.tight_layout()

plt.savefig(f'plots/figs10/stable_random_homedist_city.jpg', bbox_inches='tight', dpi=300)
##plt.savefig(f'plots/figs10/stable_random_homedist_city.pdf', bbox_inches='tight')
plt.show()

In [ ]:
dhhstat = dist_cmp[dist_cmp.city=='Gelsenkirchen'][['event','stability','dhometohome']].copy(deep=True)
dhhstat['stability'] = ['recurrent' if sta==1 else 'random' for sta in dhhstat.stability]
dhhstat['dhometohome_log'] = dhhstat.dhometohome.apply(lambda x: np.log10(1e-3+x))
print(dhhstat.dhometohome.min(), dhhstat.dhometohome.max())
dhhstat

In [ ]:
# Initialize a grid of plots with an Axes for each walk
grid = sns.FacetGrid(dhhstat, col="event", hue="stability", hue_order=['random','recurrent'], palette=sns.husl_palette(2),#sns.color_palette(),#"tab20c",
                     col_wrap=2, height=2.5, aspect=1, legend_out=True)

# Draw a line plot to show the trajectory of each random walk
grid.map(sns.histplot, "dhometohome", binwidth=.25, binrange=(-4.,3.), multiple="stack",edgecolor=".3", linewidth=.5, log_scale=True,)# binwidth=.25,
grid.set(xlim=[1e-4,1e3], xlabel="home-to-home dist [km]", ylabel="observed contacts")
grid.set(xticks=[1e-3,1e-2,1e-1,1e0,1e1,1e2])

# legend
grid.add_legend(title='category')

# Custom formatter: show decimals only when needed
def smart_format(x, _):
    return f"{x:.2f}".rstrip('0').rstrip('.') if x < 10 else f"{int(x)}"

for ax, tit in zip(grid.axes.flat, ['non-event', 'event']):
    ax.set_title(tit)
    #ax.xaxis.set_major_formatter(formatter)
    #ax.xaxis.set_major_formatter(mpl.ticker.FormatStrFormatter('%.2f'))
    #ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(smart_format))

plt.savefig(f'plots/figs10/stable_random_homehomedist_city_linear.jpg', bbox_inches='tight', dpi=300)
plt.savefig(f'plots/figs10/stable_random_homehomedist_city_linear.pdf', bbox_inches='tight')
plt.show()

In [ ]:
#stab_here = 'recurrent'
stab_here = 'random'
data1 = dhhstat[(~dhhstat.dhometohome.isna()) & (dhhstat.stability==stab_here) & (dhhstat.event==False)].dhometohome_log.tolist()
data2 = dhhstat[(~dhhstat.dhometohome.isna()) & (dhhstat.stability==stab_here) & (dhhstat.event==True)].dhometohome_log.tolist()
print(len(data1), len(data2))

print()
 
statistic, p_value = ks_2samp(data1, data2)
print(f"KS statistic: {statistic:.3f}, p-value: {p_value}")# {p_value:.4f}")

print()

statistic, p_value = mannwhitneyu(data1, data2, alternative='two-sided')
print(f"Mann–Whitney U statistic: {statistic}, p-value: {p_value}")# {p_value:.4f}")

#print()
#
#result = anderson_ksamp([data1, data2])
#print(f"AD statistic: {result.statistic}, p-value: {result.significance_level:.4f}")

In [ ]:
catstat = dist_cmp[dist_cmp.city=='Gelsenkirchen'].groupby(['day','event','stability']).pair.apply(lambda x: len(x)).reset_index().drop(columns=['day'])# len(set(x))
catstat['stability'] = ['recurrent' if sta==1 else 'random' for sta in catstat.stability]
catstat['pair'] = catstat.pair.astype(float)
catstat['x'] = catstat.stability.map({'random':1,'recurrent':0})
catstat['y'] = catstat.event.map({True:1,False:0})
catstat

In [ ]:
# Initialize a grid of plots with an Axes for each walk
grid = sns.FacetGrid(catstat, col="event", hue="stability", hue_order=['random','recurrent'], palette=sns.husl_palette(2),#sns.color_palette(),#"tab20c",
                     col_wrap=2, height=2.5, aspect=1, legend_out=True)

# Draw a line plot to show the trajectory of each random walk
grid.map(sns.violinplot, "x", "pair", orient="v",
               split=False, bw_adjust=1, cut=.1, linewidth=1, order=[0,1],
               fill=False, inner='box', gap=.25, density_norm='width',
               legend=False,
               inner_kws=dict(box_width=7.5, whis_width=0))
grid.set(xlabel="", ylabel="observed contacts")
grid.set(xticks=[0,1], xticklabels=['recurrent', 'random'])

# Custom formatter: show decimals only when needed
def smart_format(x, _):
    return f"{x:.2f}".rstrip('0').rstrip('.') if x < 10 else f"{int(x)}"

for ax, tit in zip(grid.axes.flat, ['non-event', 'event']):
    ax.set_title(tit)
    #ax.xaxis.set_major_formatter(formatter)
    #ax.xaxis.set_major_formatter(mpl.ticker.FormatStrFormatter('%.2f'))
    #ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(smart_format))

plt.show()

In [ ]:
# Initialize a grid of plots with an Axes for each walk
grid = sns.FacetGrid(catstat, col="stability", hue="stability", hue_order=['random','recurrent'], palette=sns.husl_palette(2),#sns.color_palette(),#"tab20c",
                     col_order=['recurrent','random'], col_wrap=2, height=2.5, aspect=1, legend_out=True)

# Draw a line plot to show the trajectory of each random walk
grid.map(sns.violinplot, "y", "pair", orient="v",
               split=False, bw_adjust=1, cut=.1, linewidth=1, order=[0,1],
               fill=False, inner='box', gap=.25, density_norm='width',
               legend=False,
               inner_kws=dict(box_width=7.5, whis_width=0))
grid.set(xlabel="", ylabel="observed contacts")
grid.set(xticks=[0,1], xticklabels=['non-event','event'])

# Custom formatter: show decimals only when needed
def smart_format(x, _):
    return f"{x:.2f}".rstrip('0').rstrip('.') if x < 10 else f"{int(x)}"

for ax, tit in zip(grid.axes.flat, ['stable', 'random']):
    ax.set_title(tit)
    #ax.xaxis.set_major_formatter(formatter)
    #ax.xaxis.set_major_formatter(mpl.ticker.FormatStrFormatter('%.2f'))
    #ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(smart_format))

plt.savefig(f'plots/figs10/stable_random_homedist_city_histo.jpg', bbox_inches='tight', dpi=300)
plt.savefig(f'plots/figs10/stable_random_homedist_city_histo.pdf', bbox_inches='tight')
plt.show()

In [ ]:
#stab_here = 'recurrent'
stab_here = 'random'
data1 = catstat[(catstat.stability==stab_here) & (catstat.event==False)].pair.tolist()
data2 = catstat[(catstat.stability==stab_here) & (catstat.event==True)].pair.tolist()
print(len(data1), len(data2))

print()
 
statistic, p_value = ks_2samp(data1, data2)
print(f"KS statistic: {statistic:.3f}, p-value: {p_value:.4f}")

print()

statistic, p_value = mannwhitneyu(data1, data2, alternative='two-sided')
print(f"Mann–Whitney U statistic: {statistic}, p-value: {p_value:.4f}")

#print()
#
#result = anderson_ksamp([data1, data2])
#print(f"AD statistic: {result.statistic}, p-value: {result.significance_level:.4f}")

In [ ]:
len(dist_cmp[(dist_cmp.day.isin(days_here)) & (dist_cmp.event==True) & (dist_cmp.stability==0)])

In [ ]:
len(dist_cmp[(dist_cmp.day.isin(days_here)) & (dist_cmp.event==True) & (dist_cmp.stability==1)])

In [ ]:
len(dist_cmp[(dist_cmp.day.isin(days_here)) & (dist_cmp.event==False) & (dist_cmp.stability==0)])

In [ ]:
len(dist_cmp[(dist_cmp.day.isin(days_here)) & (dist_cmp.event==False) & (dist_cmp.stability==1)])